In [ ]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import matplotlib.dates as mdates
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
from scipy.stats import chisquare, kruskal, mannwhitneyu, spearmanr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
import importlib
from pymannkendall import original_test
from statsmodels.stats.multitest import multipletests
import pycountry_convert as pc

In [2]:
importlib.reload(own)

<module 'functions' from 'c:\\Studium\\X_Masterarbeit\\Data\\Master_Thesis\\Code\\functions.py'>

Include continent in the dataset

In [33]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
continent_lookup = world[["NAME", "CONTINENT"]].rename(columns={"NAME": "Country"})
df = df.merge(continent_lookup, on="Country", how="left")
df = df.rename(columns={"CONTINENT": "Continent"})
df.head()

C:\Users\yanni\AppData\Local\Temp\ipykernel_33840\157358506.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

,id,root_id,category,description,latitude,longitude,spotted_at,spotted_by,spotted_by_type,spotted_by_name,...,Lake_Animals,Lake_Animals_Nr,Lake_Waterlevel_Changes,Lake_Waterlevel_Changes_Nr,Lake_Dry,Lake_Dry_Nr,Image_Gallery,image,geo_hash,Continent
0,16726,16726,470,Bei der Brücke am Parkplatz der Pizolbahn,47.029225,9.433322,2017-02-05 14:49:14,1996,2,Simon Meili,...,NaN,NaN,NaN,NaN,NaN,NaN,17726.0,000007/2017/02/11ba68865090752368e3a24f55058a24,u0qew9mjh6fw,Europe
1,16727,16726,470,NaN,47.029225,9.433322,2017-02-05 14:50:00,1996,2,Simon Meili,...,NaN,NaN,NaN,NaN,NaN,NaN,17727.0,000007/2017/02/2f43e32aa5f88ed53822f9a75f9155ea,u0qew9mjh6fw,Europe
2,16736,16736,469,NaN,47.398311,8.543917,2017-02-10 08:36:00,1996,2,Simon Meili,...,NaN,NaN,NaN,NaN,NaN,NaN,17736.0,000007/2017/02/912b50b8cb2200e7f0fab88a7426e920,u0qjdkt7ptw6,Europe
3,16737,16737,470,NaN,47.560411,7.589407,2017-02-11 12:36:29,2005,1,Barbara Strobl,...,NaN,NaN,NaN,NaN,NaN,NaN,17737.0,000007/2017/03/b8d2705115f82a7c34e0b39cadf7415e,u0mqsdn7qp0n,Europe
4,16740,16740,470,NaN,47.127880,8.744036,2017-02-12 10:39:27,1996,2,Simon Meili,...,NaN,NaN,NaN,NaN,NaN,NaN,17740.0,000007/2017/02/a0fe95aeebbab149d2aed0e603238b44,u0qhngr5yc9f,Europe


Now we generate a dataframe with monthly percentages of hydrological variables measured. E.g. for Switzerland, in January 2024, 32% virtual scale, 17% physical scale, 5% soil moisture, 40% plastic pollution, 1% temporary stream, 3% stream type and 2% standing water type MIGHT be observed.

In [45]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

category_cols = [
    "physical scale",
    "plastic pollution",
    "soil moisture",
    "standing water type",
    "stream type",
    "temporary stream",
    "virtual scale"
]

counts = (
    df
    .groupby(["Country", "year_month", "Category"])
    .size()
    .reset_index(name="n_obs")
)

totals = (
    df
    .groupby(["Country", "year_month"])
    .size()
    .reset_index(name="total_obs")
)

counts = counts.merge(totals, on=["Country","year_month"])
counts["percent"] = round(counts["n_obs"] / counts["total_obs"] * 100,2)

category_percentages = (
    counts
    .pivot_table(
        index=["Country","year_month"],
        columns="Category",
        values="percent",
        fill_value=0
    )
    .reset_index()
)

category_counts = (
    counts
    .pivot_table(
        index=["Country", "year_month"],
        columns="Category",
        values="n_obs",
        fill_value=0
    )
    .reset_index()
)

category_counts.columns = [f"{col}_count" if col not in ["Country", "year_month"] else col for col in category_counts.columns]

#category_percentages["year_month"] = category_percentages["year_month"].dt.to_period("M")
full_range = pd.period_range("2017-02", "2026-04", freq="M")
countries = category_percentages["Country"].unique()
full_index = pd.MultiIndex.from_product(
    [countries, full_range],
    names=["Country", "year_month"]
)

category_percentages_full = (
    category_percentages
    .set_index(["Country","year_month"])
    .reindex(full_index)
    .reset_index()
)

category_percentages_full[category_cols] = category_percentages_full[category_cols].fillna(0)
mask = category_percentages_full[category_cols].any(axis=1)
category_percentages_full["top_category"] = None
category_percentages_full.loc[mask, "top_category"] = (
    category_percentages_full.loc[mask, category_cols]
    .idxmax(axis=1)
)

# add counts
category_percentages_full = category_percentages_full.merge(
    category_counts,
    on=["Country", "year_month"],
    how="left"
)

category_percentages_full["ISO_A3"] = category_percentages_full["Country"].apply(own.get_iso3)

world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
continent_lookup = world[["NAME", "CONTINENT"]].rename(columns={"NAME": "Country", "CONTINENT": "Continent"})
category_percentages_full = category_percentages_full.merge(continent_lookup, on="Country", how="left")

C:\Users\yanni\AppData\Local\Temp\ipykernel_33840\416416197.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

In [52]:
category_percentages_full.head(12)

,Country,year_month,physical scale,plastic pollution,soil moisture,standing water type,stream type,temporary stream,virtual scale,top_category,physical scale_count,plastic pollution_count,soil moisture_count,standing water type_count,stream type_count,temporary stream_count,virtual scale_count,ISO_A3,Continent,Year
0,Afghanistan,2017-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
1,Afghanistan,2017-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
2,Afghanistan,2017-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
3,Afghanistan,2017-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
4,Afghanistan,2017-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
5,Afghanistan,2017-07,0.0,0.0,0.0,0.0,0.0,0.0,100.0,virtual scale,0.0,0.0,0.0,0.0,0.0,0.0,1.0,AFG,Asia,2017
6,Afghanistan,2017-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
7,Afghanistan,2017-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
8,Afghanistan,2017-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017
9,Afghanistan,2017-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AFG,Asia,2017


Yearly graphs that show percentages per country

In [53]:
# uses category_percentages_full

cat_cols = ["physical scale", "virtual scale", "soil moisture", "stream type", "temporary stream", "plastic pollution", "standing water type"]
count_cols = ["physical scale_count", "plastic pollution_count", "soil moisture_count", "standing water type_count", "stream type_count", "temporary stream_count", "virtual scale_count"]
colors = ["#1418fc", "#14c2fc", "#82571b", "#fc1471", "#8c14fc", "#fbff2b", "#dd6ef0"]
color_dict = dict(zip(cat_cols, colors))

# add year and aggregate over years
category_percentages_full["Year"] = category_percentages_full["year_month"].dt.year

year_country_totals = category_percentages_full.groupby(["Year", "Country"])[cat_cols].sum() # sum per country every year
year_country_pct = year_country_totals.div(year_country_totals.sum(axis=1), axis=0) * 100 # percentages per country every year

years = sorted(year_country_pct.index.get_level_values("Year").unique())

for year in years:
    df_year = year_country_pct.loc[year].sort_index(ascending=False)  # Länder alphabetisch, A unten

    fig, ax = plt.subplots(figsize=(12, len(df_year) * 0.4 + 1))
    lefts = pd.Series([0.0] * len(df_year), index=df_year.index)

    for cat, color in color_dict.items():
        vals = df_year[cat]
        ax.barh(df_year.index, vals, left=lefts, color=color, label=cat, edgecolor="none")
        lefts += vals

    ax.set_xlim(0, 100)
    ax.set_ylim(-0.5, len(df_year) - 0.5)
    ax.set_xlabel("Percentage of Observations (%)", fontsize=17)
    ax.set_title(f"Observation Categories - {year}", fontsize=20)
    ax.tick_params(axis="y", labelsize=14)
    ax.grid(axis="x", linestyle="--", color="black", alpha=0.8)

    patches = [mpatches.Patch(facecolor=color_dict[cat], label=cat) for cat in cat_cols]
    ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=14, frameon=True)

    plt.savefig(f"../Products/Hydro_Categories/Country/barchart_category_{year}.png", dpi=300, bbox_inches="tight")
    plt.close()

# total (all years)

# only take countries with 100+ observations
country_total_obs = category_percentages_full.groupby("Country")[count_cols].sum().sum(axis=1)
countries_100plus = country_total_obs[country_total_obs >= 100].index

df_total = year_country_pct.groupby("Country").mean()
df_total = df_total[df_total.index.isin(countries_100plus)].sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(12, len(df_total) * 0.4 + 1))
lefts = pd.Series([0.0] * len(df_total), index=df_total.index)

for cat, color in color_dict.items():
    vals = df_total[cat]
    ax.barh(df_total.index, vals, left=lefts, color=color, label=cat, edgecolor="none")
    lefts += vals

ax.set_xlim(0, 100)
ax.set_ylim(-0.5, len(df_total) - 0.5)
ax.set_xlabel("Percentage of Observations (%)", fontsize=17)
ax.set_title("Observation Categories - Total (All Years)", fontsize=20)
ax.tick_params(axis="y", labelsize=14)
ax.grid(axis="x", linestyle="--", color="black", alpha=0.8)

patches = [mpatches.Patch(facecolor=color_dict[cat], label=cat) for cat in cat_cols]
ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=14, frameon=True)

plt.savefig(f"../Products/Hydro_Categories/Country/barchart_category_total.png", dpi=300, bbox_inches="tight")
plt.close()

Do the same for continents

In [55]:
# uses category_percentages_full

cat_cols = ["physical scale", "virtual scale", "soil moisture", "stream type", "temporary stream", "plastic pollution", "standing water type"]
count_cols = ["physical scale_count", "plastic pollution_count", "soil moisture_count", "standing water type_count", "stream type_count", "temporary stream_count", "virtual scale_count"]
colors = ["#1418fc", "#14c2fc", "#82571b", "#fc1471", "#8c14fc", "#fbff2b", "#dd6ef0"]
color_dict = dict(zip(cat_cols, colors))

category_percentages_full["Year"] = category_percentages_full["year_month"].dt.year

year_continent_totals = category_percentages_full.groupby(["Year", "Continent"])[cat_cols].sum()
year_continent_pct = year_continent_totals.div(year_continent_totals.sum(axis=1), axis=0) * 100

years = sorted(year_continent_pct.index.get_level_values("Year").unique())

for year in years:
    df_cont = year_continent_pct.loc[year].sort_index(ascending=False)

    fig, ax = plt.subplots(figsize=(12, len(df_cont) * 0.4 + 1))
    lefts = pd.Series([0.0] * len(df_cont), index=df_cont.index)

    for cat, color in color_dict.items():
        vals = df_cont[cat]
        ax.barh(df_cont.index, vals, left=lefts, color=color, label=cat, edgecolor="none")
        lefts += vals

    ax.set_xlim(0, 100)
    ax.set_ylim(-0.5, len(df_cont) - 0.5)
    ax.set_xlabel("Percentage of Observations (%)", fontsize=17)
    ax.set_title(f"Observation Categories per Continent - {year}", fontsize=20)
    ax.tick_params(axis="y", labelsize=14)
    ax.grid(axis="x", linestyle="--", color="black", alpha=0.8)

    patches = [mpatches.Patch(facecolor=color_dict[cat], label=cat) for cat in cat_cols]
    ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=14, frameon=True)

    plt.savefig(f"../Products/Hydro_Categories/Continent/barchart_category_continent_{year}.png", dpi=300, bbox_inches="tight")
    plt.close()

# total
continents_total_obs = category_percentages_full.groupby("Continent")[count_cols].sum().sum(axis=1)
continents_100plus = continents_total_obs[continents_total_obs >= 100].index

df_cont_total = year_continent_pct.groupby("Continent").mean()
df_cont_total = df_cont_total[df_cont_total.index.isin(continents_100plus)].sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(12, len(df_cont_total) * 0.4 + 1))
lefts = pd.Series([0.0] * len(df_cont_total), index=df_cont_total.index)

for cat, color in color_dict.items():
    vals = df_cont_total[cat]
    ax.barh(df_cont_total.index, vals, left=lefts, color=color, label=cat, edgecolor="none")
    lefts += vals

ax.set_xlim(0, 100)
ax.set_ylim(-0.5, len(df_cont_total) - 0.5)
ax.set_xlabel("Percentage of Observations (%)", fontsize=17)
ax.set_title("Observation Categories per Continent - Total (All Years)", fontsize=20)
ax.tick_params(axis="y", labelsize=14)
ax.grid(axis="x", linestyle="--", color="black", alpha=0.8)

patches = [mpatches.Patch(facecolor=color_dict[cat], label=cat) for cat in cat_cols]
ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=14, frameon=True)

plt.savefig(f"../Products/Hydro_Categories/Continent/barchart_category_continent_total.png", dpi=300, bbox_inches="tight")
plt.close()

Now create monthly maps of the top category

In [ ]:
# uses category_percentages_full

world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona").to_crs("+proj=robin")
countries_with_obs = category_percentages_full["ISO_A3"].unique()

category_colors = {
    "physical scale": "#1418fc",
    "plastic pollution": "#fbff2b",
    "soil moisture": "#82571b",
    "standing water type": "#dd6ef0",
    "stream type": "#fc1471",
    "temporary stream": "#8c14fc",
    "virtual scale": "#14c2fc"
}

months = category_percentages_full["year_month"].unique()

for month in months:

    month_df = category_percentages_full[
        category_percentages_full["year_month"] == month
    ]
    
    map_df = world.merge(
        month_df,
        left_on="ADM0_A3",
        right_on="ISO_A3",
        how="left"
    )

    fig, ax = plt.subplots(1,1, figsize=(16,8))

    # Basemap --> countries without any observations ever
    world.plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    # countries without observations in this specific month
    no_obs_this_month = map_df[
        (map_df["ADM0_A3"].isin(countries_with_obs)) &
        (map_df["top_category"].isna())
    ]

    if not no_obs_this_month.empty:
        no_obs_this_month.plot(
            ax=ax,
            color="lightgrey",
            edgecolor="black",
            linewidth=0.5
        )

    #countries with observations in this specific month
    for category, color in category_colors.items():
        subset = map_df[map_df["top_category"] == category]
        if not subset.empty:
            subset.plot(
                ax=ax,
                color=color,
                edgecolor="black",
                linewidth=0.5
            )
        
    legend_order = [
        "physical scale",
        "virtual scale",
        "soil moisture",
        "stream type",
        "temporary stream",
        "plastic pollution",
        "standing water type",
        "No observations this month",
        "No observations"
    ]
    legend_handles = [
        Patch(facecolor=category_colors.get(label, "lightgrey" if label == "No observations this month" else "grey"), label=label, edgecolor="black", linewidth=0.5)
        for label in legend_order
    ]
    ax.legend(handles=legend_handles, loc="lower left", title="Category", title_fontsize=12)
    
    ax.axis("off")
    plt.title(f"Dominant CrowdWater Category Type - {month}", fontsize=15)
    plt.savefig(f"../Products/Hydro_Categories/top_category/top_category_{month}.png", dpi=300, bbox_inches="tight")
    plt.close()

And now monthly maps for each category that shows the percentage of observations per country

In [ ]:
def category_plotter(df, category):
    world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona").to_crs("+proj=robin")
    countries_with_obs = df["ISO_A3"].unique()
    months = df["year_month"].unique()

    bins = [0, 20, 40, 60, 80, 100]
    labels = ["0-20%", ">20-40%", ">40-60%", ">60-80%", ">80-100%"]
    bin_col = category + "_binned"

    cmap = get_cmap("Blues", len(labels))
    legend_handles = [
        Patch(facecolor=cmap(i / (len(labels) - 1)), label=labels[i], edgecolor="black", linewidth=0.5, )
        for i in range(len(labels))
    ]
    
    legend_handles.append(Patch(facecolor="lightgrey", edgecolor="black", linewidth=0.5, label="No observations this month"))
    legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))
    
    for month in months:
        month_df = df[df["year_month"] == month].copy()
        month_df[bin_col] = pd.cut(month_df[category], bins=bins, labels=labels, include_lowest=True)
        month_df["_codes"] = month_df[bin_col].cat.codes
        
        map_df = world.merge(
            month_df,
            left_on="ADM0_A3",
            right_on="ISO_A3",
            how="left"
        )

        fig, ax = plt.subplots(1, 1, figsize=(16, 8))
        world.plot(ax=ax, color="grey", edgecolor="black", linewidth=0.5)
        
        map_df[map_df["ADM0_A3"].isin(countries_with_obs)].plot(
            ax=ax, color="lightgrey", edgecolor="black", linewidth=0.5
        )

        subset = map_df[map_df[category] > 0].copy()
        if not subset.empty:
            subset.plot(
                column="_codes", cmap="Blues", legend=False,
                vmin=0, vmax=len(labels) - 1,
                ax=ax, edgecolor="black", linewidth=0.5
            )

        ax.legend(handles=legend_handles, loc="lower left", title=f"% of observations", title_fontsize=12)
        ax.axis("off")
        plt.title(f"Share of Observations in Category {category.title()} - {month}", fontsize=15)
        plt.savefig(f"../Products/Hydro_Categories/{category.replace(' ', '_')}/{category.replace(' ', '_')}_{month}.png", dpi=300, bbox_inches="tight")
        plt.close()

In [ ]:
# uses category_percentages_full

categories = ["physical scale",
            "virtual scale",
            "soil moisture",
            "stream type",
            "temporary stream",
            "plastic pollution",
            "standing water type"
        ]
for category in categories:
    category_plotter(category_percentages_full, category)

Are changes significant?

In [ ]:
results = []

for country in countries:
    df = year_country_pct.loc[year_country_pct.index.get_level_values("Country") == country].copy()
    years_numeric = df.index.get_level_values("Year").astype(int)
    
    if len(df) < 4:  # zu wenig Datenpunkte für sinnvollen Trend
        continue
    
    for cat in cat_cols:
        vals = df[cat].values
        
        rho, p = spearmanr(years_numeric, vals)
        if pd.isna(rho) or pd.isna(p):  # nur NaN rausfiltern
            continue
        results.append({"Country": country, "Category": cat, "rho": rho, "p-value": p})

spearman_results = pd.DataFrame(results)

_, p_corrected, _, _ = multipletests(spearman_results["p-value"], method="fdr_bh")
spearman_results["p-value-corrected"] = p_corrected
spearman_results["significant"] = spearman_results["p-value-corrected"] < 0.05

print(spearman_results[spearman_results["significant"]].sort_values("p-value-corrected"))

Purely spatially: Continental aggregation of per country category shares

In [8]:
manual_mapping = {
    "Antarctica": "Antarctica"
}

category_percentages_full["Continent"] = category_percentages_full["Country"].map(
    {c: own.country_to_continent(c) for c in category_percentages_full["Country"].unique()} | manual_mapping
)